# HW8 — Gradient boosting, MLflow tracking and an honest leaderboard

## Why this notebook exists

This is the only notebook in the repository that starts with a bigger model, and that is
deliberate: after `HW6` showed that a zoo of statistical models could not beat a constant mean, and
`HW7` showed that a 24-configuration gradient-boosting sweep could not beat "always be long SPY",
the interesting question is no longer *can boosting beat a benchmark*. It is **what a boosting
leaderboard looks like when it is built so that it cannot lie**.

Booster comparisons are easy to fake and hard to do honestly:

| Failure mode | What it looks like | Guard used here |
|---|---|---|
| Best-of-many selection | report the winner of a sweep as *the* result | sweep count passed to DSR; max **and** distribution both reported |
| Accuracy-only ranking | top-1 accuracy presented as the outcome | accuracy paired with majority-class accuracy and a net Sharpe |
| Feature importance from training data | importances that reproduce the labels | permutation importance computed **out of sample** |
| No transaction costs | gross Sharpe that vanishes at 10 bp | costs inside the return stream |
| Missing baseline | "91% accuracy!" with no reference | climatology and buy-and-hold computed first |
| Reproducibility by assertion | results quoted from memory | every run logged to MLflow and read back from the tracking store |

## What this notebook produces

1. A **model zoo** — LightGBM, XGBoost, CatBoost, scikit-learn HistGradientBoosting and a logistic
   baseline — all on one walk-forward protocol with purged folds, one shared preprocessing path and
   identical cost assumptions.
2. An **MLflow tracking store** on disk, with parameters, metrics and tags per model, and a
   leaderboard **read back from the store** rather than from in-memory variables, so the tracking is
   actually exercised rather than merely called.
3. **Out-of-sample permutation importance**, which measures what the model relies on rather than
   what it saw during training.
4. **Probability calibration** (reliability, ECE, Brier skill score) for the model that wins, which
   is what determines whether its signals are usable position sizes.
5. An honest **verdict** that follows from the run, including the possibility — established
   beforehand, not hoped for — that the best booster still fails to beat its benchmarks.

In [1]:
import json
import os
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from mlfin_core import (
    TRADING_DAYS,
    FoldSafePreprocessor,
    annualised_return,
    apply_causal_execution,
    bootstrap_sharpe_ci,
    buy_and_hold_returns,
    deflated_sharpe_ratio,
    load_ohlcv,
    make_direction_target,
    make_feature_frame,
    min_track_record_length,
    positions_to_returns,
    probabilistic_sharpe_ratio,
    profit_factor,
    purged_kfold_splits,
    sharpe_ratio,
    sortino_ratio,
    win_rate,
)

print("mlfin_core imported | TRADING_DAYS =", TRADING_DAYS)

mlfin_core imported | TRADING_DAYS = 252


## Configuration

The protocol is deliberately **identical to `HW7`** — same ticker, same features, same target,
same split point, same costs, same embargo. Holding everything fixed except the learner is the
only way an A/B comparison between LightGBM and CatBoost means anything, and it also means the
numbers here can be checked against `HW7`'s directly.

The configuration that `HW7` selected on cross-validation is reused verbatim. That is not
convenience: re-tuning per model would mean each booster gets a different number of chances to look
good, and the leaderboard would then measure tuning effort rather than model quality.

In [2]:
TICKER = "SPY"
START = "2015-01-01"
HORIZON = 1
DEAD_BAND = 0.0005
EMBARGO = 5
OOS_FRACTION = 0.20
REFIT_EVERY = 21
COST_BPS = 10.0
SR_TARGET = 1.0
RNG_SEED = 0

# The configuration HW7 selected by purged cross-validation. Reused, not re-tuned.
BASE_CONFIG = {
    "learning_rate": 0.03,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 20,
    "l2_regularization": 0.0,
    "max_iter": 150,
}
N_BOOSTERS = 4          # the number of boosting configurations that will be tried; feeds DSR

df, source = load_ohlcv(TICKER, START, allow_synthetic=True)
print(f"{TICKER}: {len(df)} bars | {df.index[0].date()} -> {df.index[-1].date()}")
print(f"data source: {source}")
if "synthetic" in str(source).lower():
    print("WARNING: synthetic fallback in use. Rankings below then describe the generator,")
    print("not a market, and must not be quoted as a market result.")
print(f"\nconfig reused from HW7: {BASE_CONFIG}")

SPY: 3066 bars | 2015-01-02 -> 2026-10-02
data source: SPY (yfinance: 2015-01-01..2026-10-05)

config reused from HW7: {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'max_iter': 150}


## Features, target, and the two benchmarks

Same construction as `HW7`. The two zero-parameter benchmarks are computed **before any model is
trained**, deliberately, so they cannot be quietly redefined later:

* **climatology** — accuracy of always predicting the training-set majority class. This is the bar
  a direction classifier has to clear to have demonstrated anything.
* **buy-and-hold** — Sharpe and CAGR of holding SPY over the identical bars, net of nothing. This
  is the bar a long/flat strategy has to clear to be worth deploying.

Anything below that writes "the model achieves X" without one of these two numbers next to it has
not been evaluated.

In [3]:
feats = make_feature_frame(df, n_lags=5, extra_indicators=True)
y = make_direction_target(df["Close"], horizon=HORIZON, deadband=DEAD_BAND, labels=(0, 1, 2))
y = y.rename("y")

data = feats.join(y, how="inner")
data = data[data["y"].notna()].copy()
close = df["Close"].reindex(data.index)

n_obs = len(data)
n_oos = int(round(n_obs * OOS_FRACTION))
X = data.drop(columns=["y"])
y = data["y"]
X_IS, y_IS = X.iloc[: n_obs - n_oos], y.iloc[: n_obs - n_oos]
X_OOS, y_OOS = X.iloc[n_obs - n_oos:], y.iloc[n_obs - n_oos:]
close_all = close.to_numpy()

CLASS_NAMES = {0: "sell", 1: "buy", 2: "hold"}
CLASS_ORDER = [0, 1, 2]
BUY_I, SELL_I = 1, 0

print(f"features            : {X.shape[1]} columns")
print(f"IS (selection only) : {len(X_IS)} bars  {X_IS.index[0].date()} -> {X_IS.index[-1].date()}")
print(f"OOS (final window)  : {len(X_OOS)} bars  {X_OOS.index[0].date()} -> {X_OOS.index[-1].date()}")

MAJORITY_CLASS = int(y_IS.value_counts().idxmax())
CLIMATOLOGY_ACC = float((y_OOS == MAJORITY_CLASS).mean())
print(f"\nclimatology accuracy (always '{CLASS_NAMES[MAJORITY_CLASS]}'): {CLIMATOLOGY_ACC:.4f}")

bh_full = buy_and_hold_returns(close)
BH_SHARPE_FULL = sharpe_ratio(bh_full)
BH_CAGR_FULL = annualised_return(bh_full)
print(f"buy-and-hold Sharpe  : {BH_SHARPE_FULL:.3f}")
print(f"buy-and-hold CAGR    : {BH_CAGR_FULL:+.4f}")
print("\nThese two lines are the leaderboard's zero row. Everything below is measured against them.")

features            : 23 columns
IS (selection only) : 2452 bars  2015-01-02 -> 2024-05-27
OOS (final window)  : 613 bars  2024-05-28 -> 2026-10-01

climatology accuracy (always 'buy'): 0.5073
buy-and-hold Sharpe  : 0.806
buy-and-hold CAGR    : +0.1319

These two lines are the leaderboard's zero row. Everything below is measured against them.


## MLflow tracking store

A SQLite-backed store at `mlflow_hw8.db`, deleted and recreated at the start of every run so the
notebook is idempotent — a tracking store that silently accumulates results across executions is a
trap, because the "best run" then depends on how many times the notebook was run.

A database backend is used rather than MLflow's legacy file store, which MLflow 3.x has placed in
maintenance mode; a deployment that cannot be queried is not a tracking system.

Each model will be logged with its parameters, its metrics, and **tags recording the protocol**.
The tags matter: a leaderboard whose rows do not say which cost assumption and which split rule
produced them is not reproducible, and the most common failure in model comparison is comparing
numbers that were measured under different conditions.

In [4]:
import mlflow
from mlflow.tracking import MlflowClient

TRACKING_DB = os.path.join(NOTEBOOK_DIR, "mlflow_hw8.db")
# Recreated every run so the "best run" cannot depend on how many times this
# notebook has been executed.
for stale in (TRACKING_DB, TRACKING_DB + ".db"):
    if os.path.isfile(stale):
        os.remove(stale)
for suffix in ("-shm", "-wal"):
    if os.path.isfile(TRACKING_DB + suffix):
        os.remove(TRACKING_DB + suffix)

# SQLite backend rather than the legacy file store: MLflow 3.x puts the filesystem
# backend in maintenance mode, and a database backend is what a real deployment uses.
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB.replace(os.sep, '/')}")
EXPERIMENT_NAME = "hw8_boosting_leaderboard"
_experiment = mlflow.set_experiment(EXPERIMENT_NAME)
# MLflow 3.x returns an Experiment object here, not a bare id.
experiment_id = _experiment.experiment_id
client = MlflowClient()

print(f"tracking uri : {mlflow.get_tracking_uri()}")
print(f"experiment   : {EXPERIMENT_NAME} (id={experiment_id})")
print(f"store exists : {os.path.isfile(TRACKING_DB)}")

2026/10/06 00:04:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/06 00:04:58 INFO mlflow.store.db.utils: Updating database tables


2026/10/06 00:05:48 INFO mlflow.tracking.fluent: Experiment with name 'hw8_boosting_leaderboard' does not exist. Creating a new experiment.


tracking uri : sqlite:///C:/Users/MV/AppData/Local/Temp/opencode/ML_Fin_Notebooks/mlflow_hw8.db
experiment   : hw8_boosting_leaderboard (id=1)
store exists : True


## The model zoo

Four boosting implementations plus a linear baseline, because the linear model is the cheapest
possible check on whether the boosters are doing anything at all.

Each entry is a **factory function**: it builds an unfitted estimator from a config dict. That
keeps refit logic identical across libraries, which matters more than it sounds — the walk-forward
loop below must not contain four slightly different hand-written training routines.

Common settings are aligned where the libraries allow it: 3 classes, fixed seed, no early stopping
on the test window. Early stopping would need a validation set carved out of the training window
*per refit*; doing it properly is possible, and it is not done here because an improper version is
the default in most examples and silently optimises on the data being scored.

In [5]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

COMMON_SEED = RNG_SEED


def make_lightgbm(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    return LGBMClassifier(
        objective="multiclass",
        num_class=3,
        learning_rate=c["learning_rate"],
        num_leaves=c["max_leaf_nodes"],
        min_child_samples=c["min_samples_leaf"],
        reg_lambda=c["l2_regularization"],
        n_estimators=c["max_iter"],
        random_state=COMMON_SEED,
        n_jobs=1,
        verbose=-1,
    )


def make_xgboost(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    # max_depth is xgboost's leaf-budget analogue; 5 is the usual match for ~15-31 leaves.
    return XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        learning_rate=c["learning_rate"],
        max_depth=5,
        min_child_weight=c["min_samples_leaf"] / 5.0,
        reg_lambda=c["l2_regularization"],
        n_estimators=c["max_iter"],
        tree_method="hist",
        random_state=COMMON_SEED,
        n_jobs=1,
        verbosity=0,
    )


def make_catboost(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    return CatBoostClassifier(
        loss_function="MultiClass",
        learning_rate=c["learning_rate"],
        depth=6,
        l2_leaf_reg=max(c["l2_regularization"], 1.0),
        iterations=c["max_iter"],
        random_seed=COMMON_SEED,
        verbose=0,
        allow_writing_files=False,
        thread_count=1,
    )


def make_sklearn_hgb(cfg=None):
    c = {**BASE_CONFIG, **(cfg or {})}
    return HistGradientBoostingClassifier(
        learning_rate=c["learning_rate"],
        max_leaf_nodes=c["max_leaf_nodes"],
        min_samples_leaf=c["min_samples_leaf"],
        l2_regularization=c["l2_regularization"],
        max_iter=c["max_iter"],
        random_state=COMMON_SEED,
    )


def make_logreg(cfg=None):
    c = cfg or {}
    return LogisticRegression(C=c.get("C", 1.0), max_iter=2000, random_state=COMMON_SEED)


ZOO = {
    "LightGBM": (make_lightgbm, BASE_CONFIG),
    "XGBoost": (make_xgboost, BASE_CONFIG),
    "CatBoost": (make_catboost, BASE_CONFIG),
    "sklearn-HGB": (make_sklearn_hgb, BASE_CONFIG),
    "LogisticRegression": (make_logreg, {"C": 1.0}),
}

print(f"{'model':<22} {'estimator':<28} params")
for name, (factory, cfg) in ZOO.items():
    est = factory()
    n_param = sum(1 for _ in est.get_params(deep=False))
    print(f"{name:<22} {type(est).__name__:<28} {n_param} constructor params")
print(f"\n{N_BOOSTERS} boosting configurations + 1 linear baseline -> "
      f"{N_BOOSTERS + 1} models for DSR")

model                  estimator                    params
LightGBM               LGBMClassifier               21 constructor params
XGBoost                XGBClassifier                41 constructor params
CatBoost               CatBoostClassifier           9 constructor params
sklearn-HGB            HistGradientBoostingClassifier 21 constructor params
LogisticRegression     LogisticRegression           14 constructor params

4 boosting configurations + 1 linear baseline -> 5 models for DSR


## Walk-forward evaluation: one protocol, five models

Every model sees the same bars in the same order with the same rules:

* Expanding training window, refit every `REFIT_EVERY` bars.
* Preprocessing statistics recomputed inside each refit.
* Each bar scored by a model fitted only on strictly earlier bars.
* Probabilities → discretionary positions with an abstention threshold and a reversal confirmation
  filter → returns with costs.

Two numbers are produced per model, and they are different questions:

* **OOS accuracy** — did it classify direction better than "always be long"? Bar = climatology.
* **OOS net Sharpe** — did it make money net of costs? Bar = buy-and-hold.

A model can pass the first and fail the second, and the leaderboard keeps them in separate columns
precisely so that this is visible rather than averaged into a single flattering score.

In [6]:
WF_START = EMBARGO + n_oos


def walk_forward_probabilities(factory, cfg) -> np.ndarray:
    """Causal walk-forward out-of-sample probabilities for the whole sample.

    Row t is filled only by a model fitted on bars [0, t), so no bar is ever
    scored by a model that saw it or anything after it.
    """
    prob = np.full((n_obs, len(CLASS_ORDER)), np.nan)
    for start in range(WF_START, n_obs, REFIT_EVERY):
        end = min(start + REFIT_EVERY, n_obs)
        tr = np.arange(0, start)
        pre = FoldSafePreprocessor().fit(X.iloc[tr])
        model = factory(cfg)
        model.fit(pre.transform(X.iloc[tr]), y.iloc[tr].to_numpy())
        proba = model.predict_proba(pre.transform(X.iloc[start:end]))
        for src_i, cls in enumerate(model.classes_):
            prob[start:end, CLASS_ORDER.index(int(cls))] = proba[:, src_i]
        prob[start:end] = np.nan_to_num(prob[start:end], nan=0.0)
    return prob


def summarise(prob: np.ndarray, name: str) -> dict:
    valid = ~np.isnan(prob).any(axis=1)
    idx = np.flatnonzero(valid)
    pos = apply_causal_execution(prob[idx], buy=BUY_I, sell=SELL_I,
                                 confirm_bars=2, min_confidence=0.40)
    net = positions_to_returns(pos, close.iloc[idx], cost_bps=COST_BPS)
    bh = buy_and_hold_returns(close.iloc[idx])
    frame = pd.DataFrame({"net": net, "bh": bh}).dropna()

    pred = prob[idx].argmax(axis=1)
    y_true = y.iloc[idx].to_numpy()
    acc = float((pred == y_true).mean())

    oos_mask = frame.index >= X_OOS.index[0]
    net_oos, bh_oos = frame.loc[oos_mask, "net"], frame.loc[oos_mask, "bh"]

    return {
        "model": name,
        "bars": int(len(idx)),
        "accuracy": acc,
        "accuracy_skill_vs_climatology": acc - CLIMATOLOGY_ACC,
        "accuracy_correct_oos": float((pred[oos_mask] == y_true[oos_mask]).mean()),
        "sharpe_net": sharpe_ratio(frame["net"]),
        "sharpe_net_oos": sharpe_ratio(net_oos),
        "sharpe_buy_hold_oos": sharpe_ratio(bh_oos),
        "sharpe_edge_vs_bh_oos": sharpe_ratio(net_oos) - sharpe_ratio(bh_oos),
        "cagr_net_oos": annualised_return(net_oos),
        "cagr_buy_hold_oos": annualised_return(bh_oos),
        "sortino_net_oos": sortino_ratio(net_oos),
        "profit_factor": profit_factor(frame["net"]),
        "win_rate": win_rate(frame["net"]),
        "trades": int((np.abs(np.diff(pos)) > 0).sum()),
        "exposure": float(np.abs(pos).mean()),
        "_net": frame["net"],
    }


results, store, net_store = [], {}, {}
for name, (factory, cfg) in ZOO.items():
    t0 = time.perf_counter()
    prob = walk_forward_probabilities(factory, cfg)
    rec = summarise(prob, name)
    rec["fit_seconds"] = time.perf_counter() - t0
    store[name] = prob
    net_store[name] = rec.pop("_net")
    results.append(rec)
    print(f"{name:<22} done in {rec['fit_seconds']:6.1f}s   "
          f"acc {rec['accuracy']:.4f}  sharpe {rec['sharpe_net']:+.3f}")

zoo_tbl = pd.DataFrame(results)
print("\n" + "=" * 100)
print("WALK-FORWARD LEADERBOARD, net of costs, identical protocol for every row")
print("=" * 100)
with pd.option_context("display.width", 200, "display.max_columns", 40):
    print(zoo_tbl[["model", "accuracy", "accuracy_skill_vs_climatology", "accuracy_correct_oos",
                   "sharpe_net_oos", "sharpe_buy_hold_oos", "sharpe_edge_vs_bh_oos",
                   "cagr_net_oos", "trades", "exposure"]].round(4).to_string(index=False))

print(f"\nreference rows:  climatology accuracy {CLIMATOLOGY_ACC:.4f} | "
      f"buy-and-hold Sharpe (OOS) {zoo_tbl['sharpe_buy_hold_oos'].iloc[0]:.3f}")
print("A model is only interesting if it is to the RIGHT of climatology on accuracy and to the")
print("RIGHT of buy-and-hold on Sharpe. Most leaderboards report only the first column.")

LightGBM               done in   59.3s   acc 0.4544  sharpe -0.315


XGBoost                done in  142.1s   acc 0.4524  sharpe -0.335


CatBoost               done in  589.0s   acc 0.4671  sharpe +0.074


sklearn-HGB            done in  143.8s   acc 0.4508  sharpe -0.201


LogisticRegression     done in    9.4s   acc 0.4810  sharpe +0.521

WALK-FORWARD LEADERBOARD, net of costs, identical protocol for every row
             model  accuracy  accuracy_skill_vs_climatology  accuracy_correct_oos  sharpe_net_oos  sharpe_buy_hold_oos  sharpe_edge_vs_bh_oos  cagr_net_oos  trades  exposure
          LightGBM    0.4544                        -0.0529                0.4600          0.1919                1.106                -0.9140        0.0181     207    0.9967
           XGBoost    0.4524                        -0.0550                0.4927         -0.3488                1.106                -1.4548       -0.0657     237    0.9959
          CatBoost    0.4671                        -0.0402                0.4829          0.6195                1.106                -0.4864        0.0895     170    1.0000
       sklearn-HGB    0.4508                        -0.0566                0.4747          0.3819                1.106                -0.7240        0.0492     235

## Logging to MLflow, then reading the leaderboard back out

The leaderboard above is built from a Python list. To prove the tracking store works, the same
metrics are logged to MLflow and the leaderboard is then **rebuilt from the tracking store** using
`MlflowClient`. If the two tables disagree, the notebook says so rather than trusting the in-memory
one.

This is a small thing to do and an unusually common thing to skip. In practice it catches real
problems: a metric logged under a different name, a run that was never written, a stale run from a
previous execution still sitting in the store.

In [7]:
PROTOCOL_TAGS = {
    "ticker": TICKER,
    "cost_bps": str(COST_BPS),
    "embargo_bars": str(EMBARGO),
    "refit_every": str(REFIT_EVERY),
    "split": "expanding walk-forward, final 20% untouched",
    "target": f"3-class direction, horizon={HORIZON}, deadband={DEAD_BAND}",
    "preprocessing": "FoldSafePreprocessor fitted inside each refit",
}

run_ids = {}
with mlflow.start_run(run_name="benchmark_buy_and_hold") as r:
    mlflow.log_metric("sharpe_buy_hold", BH_SHARPE_FULL)
    mlflow.log_metric("cagr_buy_hold", BH_CAGR_FULL)
    mlflow.set_tag("row_type", "benchmark")
    mlflow.set_tags(PROTOCOL_TAGS)
    run_ids["buy_and_hold"] = r.info.run_id

with mlflow.start_run(run_name="benchmark_climatology") as r:
    mlflow.log_metric("accuracy_climatology", CLIMATOLOGY_ACC)
    mlflow.set_tag("row_type", "benchmark")
    mlflow.set_tags(PROTOCOL_TAGS)
    run_ids["climatology"] = r.info.run_id

for name, (factory, cfg) in ZOO.items():
    rec = next(x for x in results if x["model"] == name)
    with mlflow.start_run(run_name=name) as r:
        for k, v in cfg.items():
            mlflow.log_param(k, v)
        for k, v in rec.items():
            if k not in ("model",) and isinstance(v, (int, float, np.integer, np.floating)):
                mlflow.log_metric(k, float(v))
        mlflow.set_tag("row_type", "model")
        mlflow.set_tags(PROTOCOL_TAGS)
        run_ids[name] = r.info.run_id

print(f"logged {len(run_ids)} runs to experiment '{EXPERIMENT_NAME}'")

# Rebuild the leaderboard FROM the tracking store.
rows = []
for run in client.search_runs([experiment_id]):
    m = run.data.metrics
    p = run.data.params
    row = {"run_name": run.data.tags.get("mlflow.runName"),
           "row_type": run.data.tags.get("row_type"),
           "sharpe_buy_hold": m.get("sharpe_buy_hold"),
           "accuracy_climatology": m.get("accuracy_climatology")}
    for key in ("accuracy", "accuracy_correct_oos", "sharpe_net", "sharpe_net_oos",
                "sharpe_buy_hold_oos", "sharpe_edge_vs_bh_oos", "cagr_net_oos",
                "trades", "exposure"):
        row[key] = m.get(key)
    rows.append(row)

tracked = pd.DataFrame(rows)
print("\nleaderboard reconstructed from the MLflow store:\n")
print(tracked.round(4).to_string(index=False))

model_rows = tracked[tracked["row_type"] == "model"].sort_values("sharpe_net_oos", ascending=False)
in_memory = zoo_tbl.set_index("model").loc[model_rows["run_name"].tolist(), "sharpe_net_oos"]
rebuilt = pd.Series(model_rows["sharpe_net_oos"].to_numpy(), index=model_rows["run_name"])
agree = bool(np.allclose(in_memory.to_numpy(dtype=float), rebuilt.to_numpy(dtype=float)))
print(f"\nin-memory Sharpe == store-rebuilt Sharpe : {agree}")
if not agree:
    print("MISMATCH: the tracking store and the in-memory table disagree. Do not trust the")
    print("leaderboard until the discrepancy is explained.")
else:
    print("The store round-trips, so the numbers quoted below can be recovered from MLflow alone.")

logged 7 runs to experiment 'hw8_boosting_leaderboard'

leaderboard reconstructed from the MLflow store:

              run_name  row_type  sharpe_buy_hold  accuracy_climatology  accuracy  accuracy_correct_oos  sharpe_net  sharpe_net_oos  sharpe_buy_hold_oos  sharpe_edge_vs_bh_oos  cagr_net_oos  trades  exposure
    LogisticRegression     model              NaN                   NaN    0.4810                0.4878      0.5207          0.1203                1.106                -0.9856        0.0063    66.0    0.9873
           sklearn-HGB     model              NaN                   NaN    0.4508                0.4747     -0.2006          0.3819                1.106                -0.7240        0.0492   235.0    0.9963
              CatBoost     model              NaN                   NaN    0.4671                0.4829      0.0739          0.6195                1.106                -0.4864        0.0895   170.0    1.0000
               XGBoost     model              NaN             

## Out-of-sample permutation importance

Feature importance computed on the training set answers "which features did the model use", which
is a property of the fit. The question worth answering is "which features does the model *rely on*
when it meets data it has not seen", and the honest way to measure that is to permute a feature in
held-out data and observe the damage.

This is also a leakage detector in disguise: a feature that is genuinely useful will hurt when
permuted, and a feature the model is shortcutting through will also hurt. A feature with **no**
effect out of sample, despite a high training importance, is a warning sign — the model memorised
it rather than using it.

Importance is computed on the final OOS window only, using the model refitted on IS data alone.

In [8]:
from sklearn.inspection import permutation_importance

BEST_BY_SHARPE = model_rows["run_name"].iloc[0]
print(f"model examined for importance: {BEST_BY_SHARPE}")

# Refit that model on the IS window only, then permute on the untouched OOS window.
best_factory, best_cfg = ZOO[BEST_BY_SHARPE]
pre = FoldSafePreprocessor().fit(X_IS)
X_IS_s = pre.transform(X_IS)
X_OOS_s = pre.transform(X_OOS)
imp_model = best_factory(best_cfg)
imp_model.fit(X_IS_s, y_IS.to_numpy())

perm = permutation_importance(
    imp_model, X_OOS_s, y_OOS.to_numpy(),
    scoring="accuracy", n_repeats=20, random_state=RNG_SEED, n_jobs=1)

imp = pd.DataFrame({
    "feature": X_OOS.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
imp["pct_of_total_loss"] = 100.0 * imp["importance_mean"].clip(lower=0) / \
    max(imp["importance_mean"].clip(lower=0).sum(), 1e-12)

print(f"\nout-of-sample permutation importance (accuracy units, {len(X_OOS)} OOS bars)\n")
print(imp.head(12).round(5).to_string(index=False))

zero_effect = int((imp["importance_mean"].abs() < 1e-4).sum())
print(f"\nfeatures with no measurable OOS effect : {zero_effect} / {len(imp)}")
print(f"top-3 features carry                 : "
      f"{imp['pct_of_total_loss'].head(3).sum():.1f}% of the measured effect")
if zero_effect > len(imp) // 2:
    print("Most features do nothing out of sample. A feature set this flat cannot support a")
    print("model with many effective degrees of freedom, which is consistent with the")
    print("leaderboard above and with HW7's result.")

model examined for importance: CatBoost



out-of-sample permutation importance (accuracy units, 613 OOS bars)

    feature  importance_mean  importance_std  pct_of_total_loss
  close_loc          0.00922         0.00906           17.54658
  ret_lag_2          0.00889         0.00848           16.92547
  ret_lag_4          0.00726         0.00793           13.81988
   ma_ratio          0.00546         0.00767           10.40373
  macd_hist          0.00498         0.00660            9.47205
  vol_ratio          0.00465         0.00759            8.85093
macd_signal          0.00432         0.00651            8.22981
  ret_lag_5          0.00375         0.00597            7.14286
     atr_14          0.00367         0.00565            6.98758
  ret_lag_1          0.00033         0.00729            0.62112
Close_Lag_4          0.00000         0.00000            0.00000
   ema_fast          0.00000         0.00000            0.00000

features with no measurable OOS effect : 6 / 23
top-3 features carry                 : 48.3% of t

## Calibration of the winning model

A classifier can be directionally useful and still be useless as a position sizer. `HW6` showed
that interval coverage is not conditional on the state of the world; this cell asks the simpler,
more directly usable question: when the model says 70% buy, how often is it right?

Three diagnostics, all computed on the OOS window:

* **Reliability table** — predicted probability against observed frequency, per bin.
* **Expected calibration error (ECE)** — sample-weighted mean absolute gap, equal-width bins.
* **Brier skill score** — against the climatology forecast, so it inherits the same benchmark
  discipline as everything else.

If the model's edge is not large, these will look like the prior — which is itself the honest
result and should be reported as such rather than hidden behind a single accuracy figure.

In [9]:
from mlfin_core import brier_skill_score, expected_calibration_error, reliability_table

best_prob = store[BEST_BY_SHARPE]
best_valid = ~np.isnan(best_prob).any(axis=1)
oos_mask_np = np.asarray(data.index >= X_OOS.index[0])
best_idx = np.flatnonzero(best_valid & oos_mask_np)
oos_prob = best_prob[best_idx]
oos_true = y.iloc[best_idx].to_numpy()

calib_pred = oos_prob[:, BUY_I]
reliability = reliability_table(y_true=oos_true, y_prob=oos_prob[:, BUY_I], n_bins=5)
reliability.columns = [c.replace("y_true", "observed_buy") for c in reliability.columns]
print(reliability.round(4).to_string())

ECE = expected_calibration_error(y_true=oos_true, y_prob=oos_prob[:, BUY_I], n_bins=10)

# Multiclass Brier needs the full (n, 3) probability matrix, and the reference
# forecast must come from the IS window only -- computing it from the test
# frequencies would quietly make the benchmark easier than the model.
class_prior = (
    y.iloc[: n_obs - n_oos].value_counts(normalize=True)
    .reindex(CLASS_ORDER).fillna(0.0).to_numpy(dtype=float)
)
prior = float(class_prior[BUY_I])
clim_matrix = np.tile(class_prior, (len(oos_true), 1))
BSS = brier_skill_score(y_true=oos_true, y_prob=oos_prob, climatology=clim_matrix)

print(f"\nbuy-class prior on IS               : {prior:.4f}")
print(f"observed buy frequency on OOS       : {float((oos_true == BUY_I).mean()):.4f}")
print(f"mean predicted buy probability      : {calib_pred.mean():.4f}")
print(f"expected calibration error (10 bins): {ECE:.4f}")
print(f"multiclass Brier skill vs IS climatology : {BSS:+.4f}")
print(f"mean predicted max probability      : {oos_prob.max(axis=1).mean():.4f}")
print(f"bars where max prob >= 0.40         : "
      f"{float((oos_prob.max(axis=1) >= 0.40).mean()):.1%}")

CALIBRATED_ENOUGH = bool(np.isfinite(BSS) and BSS > 0.0 and ECE < 0.10)
print(f"\nuseful as a position sizer          : {CALIBRATED_ENOUGH}")
if not CALIBRATED_ENOUGH:
    print("Brier skill <= 0 or ECE >= 0.10 means the probabilities are no better than the")
    print("climatology forecast. Positions sized off them would add cost and risk, not edge.")

   bin    n  pred_mean  obs_freq     gap
0    1   13     0.3897    0.8462  0.4565
1    2  586     0.5002    0.7287  0.2285
2    3   14     0.6150    0.5000 -0.1150

buy-class prior on IS               : 0.4898
observed buy frequency on OOS       : 0.5073
mean predicted buy probability      : 0.5004
expected calibration error (10 bins): 0.2308
multiclass Brier skill vs IS climatology : -0.0077
mean predicted max probability      : 0.5056
bars where max prob >= 0.40         : 99.8%

useful as a position sizer          : False
Brier skill <= 0 or ECE >= 0.10 means the probabilities are no better than the
climatology forecast. Positions sized off them would add cost and risk, not edge.


## Uncertainty on the leaderboard's best row

`HW6` and `HW7` both ended with the same discipline, and it carries over unchanged: a point estimate
from a few hundred bars is a claim about noise.

* **Block bootstrap** confidence interval on the net Sharpe of the winning model.
* **PSR** against zero.
* **DSR** against the best of `N_BOOSTERS + 1` trials — the honest benchmark when four boosting
  implementations and a linear model were all tried.
* **MinTRL** — how long a track record would be needed to establish the target Sharpe at all.

If the winner's interval includes zero, the leaderboard has not found an edge; it has found the
least-bad member of a small family.

In [10]:
# The winning model's net return stream, kept from the evaluation pass rather than
# recomputed, so the uncertainty is measured on exactly the series that was reported.
best_net = net_store[BEST_BY_SHARPE]

ci = bootstrap_sharpe_ci(best_net, n_boot=4000, block_size=10, alpha=0.05, seed=RNG_SEED)
psr = probabilistic_sharpe_ratio(best_net, sr_benchmark=0.0)
dsr = deflated_sharpe_ratio(best_net, n_trials=N_BOOSTERS + 1)
mintrl = min_track_record_length(best_net, sr_target=SR_TARGET)
MINTRL_OK = bool(np.isfinite(mintrl))

print(f"uncertainty on {BEST_BY_SHARPE}")
print(f"  net Sharpe point estimate        : {ci['point']:.3f}")
print(f"  95% block-bootstrap CI           : [{ci['lo']:.3f}, {ci['hi']:.3f}]")
print(f"  P(Sharpe > 0)                   : {ci['p_gt_zero']:.3f}")
print(f"  PSR vs zero                     : {psr:.4f}")
print(f"  DSR vs best of {N_BOOSTERS + 1} trials       : {dsr['dsr']:.4f} "
      f"(SR benchmark {dsr['sr_benchmark']:.3f})")
if MINTRL_OK:
    print(f"  MinTRL for SR {SR_TARGET:.1f}             : {mintrl / TRADING_DAYS:.1f} years "
          f"vs {len(best_net) / TRADING_DAYS:.1f} observed")
else:
    print(f"  MinTRL for SR {SR_TARGET:.1f}             : undefined (realised Sharpe <= 0)")

CI_CLEARS = bool(np.isfinite(ci["lo"]) and ci["lo"] > 0)
DSR_CLEARS = bool(np.isfinite(dsr["dsr"]) and dsr["dsr"] >= 0.95)
VERDICT = "credible edge" if (CI_CLEARS and DSR_CLEARS) else "no credible edge"

print(f"\n  95% CI excludes zero : {CI_CLEARS}")
print(f"  DSR >= 0.95          : {DSR_CLEARS}")
print(f"  VERDICT              : {VERDICT}")

BEATS_CLIMATOLOGY = bool(zoo_tbl["accuracy"].max() > CLIMATOLOGY_ACC)
BEATS_BUY_HOLD = bool(zoo_tbl["sharpe_net_oos"].max() > zoo_tbl["sharpe_buy_hold_oos"].iloc[0])
print(f"\n  best accuracy beats climatology   : {BEATS_CLIMATOLOGY} "
      f"({zoo_tbl['accuracy'].max():.4f} vs {CLIMATOLOGY_ACC:.4f})")
print(f"  best Sharpe beats buy-and-hold    : {BEATS_BUY_HOLD} "
      f"({zoo_tbl['sharpe_net_oos'].max():.3f} vs {zoo_tbl['sharpe_buy_hold_oos'].iloc[0]:.3f})")
print("\nBoth are reported because they usually disagree, and when they do the accuracy result")
print("is the one that gets written up while the Sharpe result is the one that matters.")

uncertainty on CatBoost
  net Sharpe point estimate        : 0.074
  95% block-bootstrap CI           : [-0.485, 0.673]
  P(Sharpe > 0)                   : 0.604
  PSR vs zero                     : 0.5910
  DSR vs best of 5 trials       : 0.2044 (SR benchmark 0.021)
  MinTRL for SR 1.0             : 183.1 years vs 9.7 observed

  95% CI excludes zero : False
  DSR >= 0.95          : False
  VERDICT              : no credible edge

  best accuracy beats climatology   : False (0.4810 vs 0.5073)
  best Sharpe beats buy-and-hold    : False (0.620 vs 1.106)

Both are reported because they usually disagree, and when they do the accuracy result
is the one that gets written up while the Sharpe result is the one that matters.


## Conclusions

The tables above are printed by the code and computed from this run; the reasoning is kept separate
so the two can be checked against each other.

**1. A leaderboard is a statement about a search, not about a model.** Four boosting
implementations and a linear baseline were tried. The DSR cell is told that number, so the winner's
Sharpe is judged against the best of five attempts rather than against zero. A leaderboard that
reports only the top row has silently reported the maximum of a random variable and called it an
estimate.

**2. Accuracy and tradability are different claims, and this repository now reports both.** The
accuracy column is measured against the majority-class benchmark and the Sharpe column against
buy-and-hold over identical bars. Where a model clears the first and fails the second, it has
learned the base rate of an upward-drifting index — which is a true statement and a useless one.

**3. The benchmarks were computed before the models, on purpose.** The climatology accuracy and the
buy-and-hold Sharpe appear above every model row, with their own MLflow runs. Defining a baseline
after seeing the results is the most common way a comparison becomes unfalsifiable, and it costs
nothing to avoid.

**4. Importance is measured on data the model has not seen.** Training-set importance answers a
question about the fit; permutation importance on the OOS window answers a question about the
model. The number of features with no measurable out-of-sample effect is reported alongside the
ranking, because a feature set where most columns do nothing is the real constraint on model
complexity here.

**5. Calibration decides whether probabilities can size positions.** A model whose Brier skill
score against climatology is not positive is not producing information the risk manager can use,
however good its accuracy looks. `HW6` established the same point for intervals; here it is
established for the probabilities that generate them.

**6. The tracking store is part of the result.** Every row above is recoverable from MLflow with
its parameters, its cost assumption and its split rule attached, and the in-memory and store-rebuilt
leaderboards are compared rather than assumed equal. Reproducibility that is asserted in prose is
not reproducibility.

In [11]:
print("=" * 96)
print("BOTTOM LINE, computed from this run")
print("=" * 96)

print("\nData and protocol")
print(f"  ticker / source        : {TICKER} / {source}")
print(f"  bars / features        : {n_obs} / {X.shape[1]}")
print(f"  IS / OOS               : {len(X_IS)} / {len(X_OOS)}")
print(f"  cost per round trip    : {COST_BPS:.0f} bp")
print(f"  config reused from HW7 : {BASE_CONFIG}")

print("\nZero-parameter benchmarks (defined before any model ran)")
print(f"  climatology accuracy   : {CLIMATOLOGY_ACC:.4f} (always '{CLASS_NAMES[MAJORITY_CLASS]}')")
print(f"  buy-and-hold Sharpe    : {BH_SHARPE_FULL:.3f}  (full sample)")
print(f"  buy-and-hold CAGR      : {BH_CAGR_FULL:+.4f}")
bh_oos_ref = float(zoo_tbl['sharpe_buy_hold_oos'].iloc[0])
print(f"  buy-and-hold Sharpe    : {bh_oos_ref:.3f}  (OOS window, the comparison used above)")

print("\nLeaderboard, sorted by OOS net Sharpe")
print(f"  {'model':<22}{'acc':>8}{'vs clim':>10}{'acc OOS':>9}"
      f"{'SR OOS':>9}{'SR bh-hold':>12}{'edge':>9}{'trades':>8}")
for _, r in model_rows.iterrows():
    print(f"  {r['run_name']:<22}{r['accuracy']:>8.4f}"
          f"{r['accuracy'] - CLIMATOLOGY_ACC:>10.4f}{r['accuracy_correct_oos']:>9.4f}"
          f"{r['sharpe_net_oos']:>9.3f}{r['sharpe_buy_hold_oos']:>12.3f}"
          f"{r['sharpe_edge_vs_bh_oos']:>9.3f}{int(r['trades']):>8}")
print(f"  {'climatology':<22}{CLIMATOLOGY_ACC:>8.4f}{'--':>10}")
print(f"  {'buy-and-hold':<22}{'--':>8}{'--':>10}{'--':>9}{bh_oos_ref:>9.3f}{'--':>12}"
      f"{'0.000':>9}{'--':>8}")

print("\nRank stability")
print(f"  best by accuracy   : {zoo_tbl.loc[zoo_tbl['accuracy'].idxmax(), 'model']}")
print(f"  best by OOS Sharpe : {BEST_BY_SHARPE}")
print(f"  same model wins both: "
      f"{zoo_tbl.loc[zoo_tbl['accuracy'].idxmax(), 'model'] == BEST_BY_SHARPE}")
print(f"  OOS accuracy range across models : "
      f"{zoo_tbl['accuracy_correct_oos'].min():.4f} -> {zoo_tbl['accuracy_correct_oos'].max():.4f}")
print(f"  OOS Sharpe range across models   : "
      f"{zoo_tbl['sharpe_net_oos'].min():.3f} -> {zoo_tbl['sharpe_net_oos'].max():.3f}")

print("\nOut-of-sample permutation importance")
print(f"  model                : {BEST_BY_SHARPE}")
print(f"  top feature          : {imp.iloc[0]['feature']} "
      f"({imp.iloc[0]['importance_mean']:+.5f})")
print(f"  top-3 share of effect: {imp['pct_of_total_loss'].head(3).sum():.1f}%")
print(f"  features with no OOS effect: {zero_effect} / {len(imp)}")

print("\nCalibration on the OOS window")
print(f"  buy prior / observed : {prior:.4f} / {float((oos_true == BUY_I).mean()):.4f}")
print(f"  mean predicted buy   : {calib_pred.mean():.4f}")
print(f"  ECE (10 bins)        : {ECE:.4f}")
print(f"  multiclass BSS vs IS climatology: {BSS:+.4f}")
print(f"  usable as position sizer: {CALIBRATED_ENOUGH}")

print("\nUncertainty on the winning row")
print(f"  95% Sharpe CI        : [{ci['lo']:.3f}, {ci['hi']:.3f}]")
print(f"  PSR vs zero          : {psr:.4f}")
print(f"  DSR vs best of {N_BOOSTERS + 1}     : {dsr['dsr']:.4f}")
print(f"  MinTRL for SR {SR_TARGET:.1f}       : "
      + (f"{mintrl / TRADING_DAYS:.1f} years" if MINTRL_OK else "undefined (Sharpe <= 0)"))
print(f"  beats climatology    : {BEATS_CLIMATOLOGY}")
print(f"  beats buy-and-hold   : {BEATS_BUY_HOLD}")
print(f"\n  VERDICT              : {VERDICT}")

BOTTOM LINE, computed from this run

Data and protocol
  ticker / source        : SPY / SPY (yfinance: 2015-01-01..2026-10-05)
  bars / features        : 3065 / 23
  IS / OOS               : 2452 / 613
  cost per round trip    : 10 bp
  config reused from HW7 : {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'max_iter': 150}

Zero-parameter benchmarks (defined before any model ran)
  climatology accuracy   : 0.5073 (always 'buy')
  buy-and-hold Sharpe    : 0.806  (full sample)
  buy-and-hold CAGR      : +0.1319
  buy-and-hold Sharpe    : 1.106  (OOS window, the comparison used above)

Leaderboard, sorted by OOS net Sharpe
  model                      acc   vs clim  acc OOS   SR OOS  SR bh-hold     edge  trades
  CatBoost                0.4671   -0.0402   0.4829    0.620       1.106   -0.486     170
  sklearn-HGB             0.4508   -0.0566   0.4747    0.382       1.106   -0.724     235
  LightGBM                0.4544   -0.0529   0.4600 

In [12]:
summary = {
    "run": {
        "ticker": TICKER,
        "data_source": str(source),
        "n_bars": int(n_obs),
        "n_features": int(X.shape[1]),
        "is_bars": int(len(X_IS)),
        "oos_bars": int(len(X_OOS)),
        "cost_bps": COST_BPS,
        "embargo": EMBARGO,
        "refit_every": REFIT_EVERY,
        "config_reused_from_hw7": BASE_CONFIG,
        "n_boosters_tried": N_BOOSTERS,
    },
    "benchmarks": {
        "climatology_accuracy": CLIMATOLOGY_ACC,
        "buy_hold_sharpe_full": BH_SHARPE_FULL,
        "buy_hold_cagr_full": BH_CAGR_FULL,
        "buy_hold_sharpe_oos": bh_oos_ref,
    },
    "leaderboard": json.loads(model_rows.to_json(orient="records")),
    "mlflow": {
        "tracking_uri": mlflow.get_tracking_uri(),
        "experiment": EXPERIMENT_NAME,
        "experiment_id": experiment_id,
        "n_runs_logged": len(run_ids),
        "run_ids": run_ids,
        "in_memory_matches_store": agree,
    },
    "rank_stability": {
        "best_by_accuracy": str(zoo_tbl.loc[zoo_tbl["accuracy"].idxmax(), "model"]),
        "best_by_oos_sharpe": BEST_BY_SHARPE,
        "same_model_wins_both": bool(
            zoo_tbl.loc[zoo_tbl["accuracy"].idxmax(), "model"] == BEST_BY_SHARPE),
        "oos_accuracy_min": float(zoo_tbl["accuracy_correct_oos"].min()),
        "oos_accuracy_max": float(zoo_tbl["accuracy_correct_oos"].max()),
        "oos_sharpe_min": float(zoo_tbl["sharpe_net_oos"].min()),
        "oos_sharpe_max": float(zoo_tbl["sharpe_net_oos"].max()),
    },
    "permutation_importance": {
        "model": BEST_BY_SHARPE,
        "computed_on": "final OOS window only",
        "top_features": imp.head(10)[["feature", "importance_mean"]].to_dict("records"),
        "features_with_no_oos_effect": zero_effect,
        "n_features": int(len(imp)),
    },
    "calibration": {
        "buy_prior_is": prior,
        "buy_frequency_oos": float((oos_true == BUY_I).mean()),
        "mean_predicted_buy": float(calib_pred.mean()),
        "expected_calibration_error": ECE,
        "brier_skill_score_vs_climatology": BSS,
        "usable_as_position_sizer": CALIBRATED_ENOUGH,
    },
    "uncertainty": {
        "sharpe_ci_lo": ci["lo"],
        "sharpe_ci_hi": ci["hi"],
        "p_sharpe_gt_zero": ci["p_gt_zero"],
        "psr": psr,
        "dsr": dsr["dsr"],
        "dsr_n_trials": N_BOOSTERS + 1,
        "dsr_sr_benchmark": dsr["sr_benchmark"],
        "mintrl_years": (mintrl / TRADING_DAYS) if MINTRL_OK else None,
        "ci_excludes_zero": CI_CLEARS,
        "dsr_credible": DSR_CLEARS,
        "beats_climatology": BEATS_CLIMATOLOGY,
        "beats_buy_and_hold": BEATS_BUY_HOLD,
        "verdict": VERDICT,
    },
}
print(json.dumps(summary, indent=2, default=str))

{
  "run": {
    "ticker": "SPY",
    "data_source": "SPY (yfinance: 2015-01-01..2026-10-05)",
    "n_bars": 3065,
    "n_features": 23,
    "is_bars": 2452,
    "oos_bars": 613,
    "cost_bps": 10.0,
    "embargo": 5,
    "refit_every": 21,
    "config_reused_from_hw7": {
      "learning_rate": 0.03,
      "max_leaf_nodes": 15,
      "min_samples_leaf": 20,
      "l2_regularization": 0.0,
      "max_iter": 150
    },
    "n_boosters_tried": 4
  },
  "benchmarks": {
    "climatology_accuracy": 0.5073409461663948,
    "buy_hold_sharpe_full": 0.8064686639795501,
    "buy_hold_cagr_full": 0.1319134741460859,
    "buy_hold_sharpe_oos": 1.10595620537609
  },
  "leaderboard": [
    {
      "run_name": "CatBoost",
      "row_type": "model",
      "sharpe_buy_hold": null,
      "accuracy_climatology": null,
      "accuracy": 0.4671025746,
      "accuracy_correct_oos": 0.4828711256,
      "sharpe_net": 0.0739395251,
      "sharpe_net_oos": 0.6195299068,
      "sharpe_buy_hold_oos": 1.1059562054